# Aidoc Algorithm Evaluation

Notebook aligned with the Streamlit dashboard tabs:
**Overview → Timelines → Algo Performance**.

### First-time setup
1. Clone/download this repo and open a terminal **in this folder** (same directory as `main.ipynb` and the `.xlsx` data file).
2. Install dependencies: `pip install -r requirements.txt`
3. Open `main.ipynb` in Jupyter, VS Code, or Cursor.
4. Run **Run All** (top to bottom). The appendix cell at the bottom defines helpers loaded automatically in the data cell.

**Data file required:** `AI_data_analysis_exercise_(4)_(2)_(2)_(4).xlsx` must sit next to this notebook.


In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import display, Markdown

# Plotly renderer: works in Jupyter Lab, Notebook, and VS Code
for _renderer in ("notebook_connected", "iframe", "png"):
    try:
        pio.renderers.default = _renderer
        break
    except Exception:
        continue


In [2]:
import json
from pathlib import Path

DATA_FILENAME = "AI_data_analysis_exercise_(4)_(2)_(2)_(4).xlsx"
NOTEBOOK_FILENAME = "main.ipynb"


def find_project_root() -> Path:
    """Find repo folder containing this notebook and the Excel data file."""
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / NOTEBOOK_FILENAME).exists() and (path / DATA_FILENAME).exists():
            return path
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / NOTEBOOK_FILENAME).exists():
            return path
    return Path.cwd()


def load_helpers(notebook_path: Path) -> None:
    """Load helper definitions from the appendix cell at the bottom of this notebook."""
    if not notebook_path.exists():
        raise FileNotFoundError(
            f"Could not find {NOTEBOOK_FILENAME}. Open the project folder and run from there."
        )
    nb_local = json.loads(notebook_path.read_text(encoding="utf-8"))
    for cell in reversed(nb_local.get("cells", [])):
        src = "".join(cell.get("source", []))
        if cell.get("cell_type") == "code" and src.lstrip().startswith("COLOR_MAP"):
            exec(src, globals())
            return
    raise RuntimeError("Appendix helpers cell not found at the bottom of main.ipynb.")


ROOT = find_project_root()
NOTEBOOK_PATH = ROOT / NOTEBOOK_FILENAME
DATA_FILE = ROOT / DATA_FILENAME

load_helpers(NOTEBOOK_PATH)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Missing data file: {DATA_FILE}\n"
        f"Download or copy {DATA_FILENAME} into: {ROOT}"
    )


def load_data(path: Path = DATA_FILE) -> pd.DataFrame:
    df = pd.read_excel(path)
    for col in [
        "scan_timestamp", "radiologist_sign_time", "algos_start_run",
        "algo1_finish_run", "algo2_finish_run", "algo3_finish_run",
    ]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])
    return df


df = load_data()
filtered_df = df.copy()
print(f"Project root: {ROOT}")
print(f"Loaded {len(df):,} scans from {DATA_FILE.name}")


Project root: /Users/rave/Documents/GitHub/aidoc_algo_eval
Loaded 20,000 scans from AI_data_analysis_exercise_(4)_(2)_(2)_(4).xlsx


## Tab 1 — Overview

In [3]:
vol_df = volume_sunburst_df(filtered_df)
age_vol_df = age_prevalence_sunburst_df(filtered_df)
display(vol_df)
prevalence_sunburst(vol_df, ["site", "patient_class", "gender"], "Scan Volume", compact=True, show_title=False).show()
display(age_vol_df)
prevalence_sunburst(age_vol_df, ["patient_class", "age_group"], "Prevalence by Age", sort_slices=False, compact=True, show_title=False).show()

,site,patient_class,gender,scans,positives,prevalence_pct
0,best_doctors,ED,female,602,34,5.6
1,best_doctors,ED,male,1047,58,5.5
2,best_doctors,IN,female,2221,254,11.4
3,best_doctors,IN,male,2116,251,11.9
4,healthy_vibes,ED,female,1377,67,4.9
5,healthy_vibes,ED,male,2502,102,4.1
6,healthy_vibes,IN,female,5020,543,10.8
7,healthy_vibes,IN,male,5115,595,11.6


,patient_class,age_group,scans,positives,prevalence_pct
0,ED,00-17,249,14,5.6
1,ED,18-39,1552,87,5.6
2,ED,40-64,2489,126,5.1
3,ED,65+,1238,34,2.7
4,IN,00-17,696,94,13.5
5,IN,18-39,4084,575,14.1
6,IN,40-64,6509,755,11.6
7,IN,65+,3183,219,6.9


In [4]:
gender_df = group_distribution_df(filtered_df, "gender")
display(gender_df)
age_distribution_by_gender_chart(filtered_df).show()

,gender,scans,positives,prevalence_pct
0,female,9220,898,9.7
1,male,10780,1006,9.3


In [5]:
temporal_base = build_temporal_base(filtered_df)
hourly_total_overall = hourly_total_scans(temporal_base)
fig_tvm = go.Figure(go.Bar(
    x=hourly_total_overall["hour"], y=hourly_total_overall["scans"],
    marker_color="#6a994e", opacity=BAR_FILL_OPACITY,
    text=[f"{int(v):,}" for v in hourly_total_overall["scans"]], textposition="outside", textfont=dict(size=9),
))
mean_val = hourly_total_overall["scans"].mean()
fig_tvm.add_hline(y=mean_val, line_dash="dash", line_color="#bc4749", annotation_text=f"Mean: {mean_val:.0f}")
fig_tvm.update_layout(title="Hourly Total Scan Volume (All Scans)", xaxis_title="Hour (0-23)", yaxis_title="Total Scans", margin=dict(t=50))
fig_tvm.show()

hourly_pos_overall = hourly_pos_rate(temporal_base)
fig_prm = go.Figure(go.Scatter(x=hourly_pos_overall["hour"], y=hourly_pos_overall["pos_rate"], mode="lines+markers", marker_color="#6a994e"))
mean_pr = hourly_pos_overall["pos_rate"].mean()
fig_prm.add_hline(y=mean_pr, line_dash="dash", line_color="#bc4749", annotation_text=f"Mean: {mean_pr:.1f}%")
fig_prm.update_layout(title="Hourly Positive Rate (All Scans)", xaxis_title="Hour (0-23)", yaxis_title="Positive Rate (%)", margin=dict(t=50))
fig_prm.show()

monthly_pos_overall = monthly_pos_rate(temporal_base)
fig_mpm = go.Figure(go.Scatter(x=monthly_pos_overall["month_name"], y=monthly_pos_overall["pos_rate"], mode="lines+markers", marker_color="#6a994e"))
mean_mp = monthly_pos_overall["pos_rate"].mean()
fig_mpm.add_hline(y=mean_mp, line_dash="dash", line_color="#bc4749", annotation_text=f"Mean: {mean_mp:.1f}%")
fig_mpm.update_layout(title="Monthly Positive Rate (All Scans)", xaxis_title="Month", yaxis_title="Positive Rate (%)", margin=dict(t=50))
fig_mpm.show()

heatmap_df = heatmap_data(temporal_base)
dow_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
heatmap_df["day_name"] = heatmap_df["dow"].map(DOW_LABELS)
pivot = heatmap_df.pivot(index="day_name", columns="hour", values="pos_rate").reindex(dow_order)
hottest_idx = heatmap_df.loc[heatmap_df["pos_rate"].idxmax()]
coldest_idx = heatmap_df.loc[heatmap_df["pos_rate"].idxmin()]
fig_heat = px.imshow(pivot, aspect="auto", title="Positive Rate (%) — Day × Hour", labels={"x": "Hour", "y": "Day", "color": "Pos Rate (%)"}, color_continuous_scale="RdYlGn_r")
fig_heat.update_layout(height=400, margin=dict(t=60, b=40, l=60, r=20), coloraxis_colorbar=dict(title="Rate %", thickness=12))
fig_heat.add_annotation(x=int(hottest_idx["hour"]), y=DOW_LABELS[int(hottest_idx["dow"])], text="🔥", showarrow=False, font=dict(size=18))
fig_heat.add_annotation(x=int(coldest_idx["hour"]), y=DOW_LABELS[int(coldest_idx["dow"])], text="❄️", showarrow=False, font=dict(size=18))
fig_heat.show()

print("Note: This analysis can be more interesting when dealing with greater amount of scans / data, since the partition is quite big.")


Note: This analysis can be more interesting when dealing with greater amount of scans / data, since the partition is quite big.


## Tab 2 — Timelines

In [6]:
duration_df = compute_duration_df(filtered_df)
print(f"Valid TAT rows: {len(duration_df):,}")
if duration_df.empty:
    print("No valid TAT rows.")
else:
    tat_long = duration_df.melt(var_name="Entity", value_name="Minutes")
    tat_means = duration_df.mean()
    fig_box = px.box(tat_long, x="Entity", y="Minutes", color="Entity", color_discrete_map=COLOR_MAP, title="TAT Distribution (minutes)", points=False)
    for trace in fig_box.data:
        trace.boxmean = True
        trace.hovertemplate = f"<b>{trace.name}</b><br>Mean: {tat_means[trace.name]:.2f} min<br>Median: %{{median:.2f}} min<extra></extra>"
    fig_box.update_layout(showlegend=False, xaxis_title="")
    fig_box.show()
    display(pd.DataFrame({"Entity": duration_df.columns, "Mean (min)": duration_df.mean().round(2).values, "Median (min)": duration_df.median().round(2).values}))
    tat_dot_plot(duration_df).show()
    tat_win_count_chart(duration_df).show()
    fig_clin, mean_gap, median_gap = tat_clinical_gap_chart(duration_df)
    print(f"Mean gap (Rad − Algo 3): {mean_gap:.2f} min | Median: {median_gap:.2f} min")
    fig_clin.show()

Valid TAT rows: 19,982


,Entity,Mean (min),Median (min)
0,Radiologist,9.73,9.65
1,Algo 1,0.75,0.75
2,Algo 2,2.50,2.50
3,Algo 3,10.02,10.03


Mean gap (Rad − Algo 3): -0.29 min | Median: -0.30 min


In [7]:
hourly_gender_tat = hourly_scans_rad_by_group(filtered_df, "gender")
hourly_dept_tat = hourly_scans_rad_by_group(filtered_df, "patient_class")
hourly_scans_rad_chart(hourly_gender_tat, "gender", GENDER_COLOR_MAP, "Hourly Scan Volume & Radiologist Time — by Gender").show()
hourly_scans_rad_chart(hourly_dept_tat, "patient_class", DEPT_COLOR_MAP, "Hourly Scan Volume & Radiologist Time — by Department").show()
hourly_dept_algo = hourly_scans_algo_by_group(filtered_df, "patient_class")
hourly_scans_algo_chart(hourly_dept_algo, "patient_class").show()

## Tab 3 — Algo Performance

In [8]:
metrics_df = compute_metrics_df(filtered_df)
display(metrics_df)
confusion_matrices_chart(metrics_df).show()
diagnostic_radar_chart(metrics_df).show()
f1_score_chart(metrics_df).show()
npv_chart(metrics_df).show()
error_rate_chart(metrics_df).show()

,Algorithm,TP,TN,FP,FN,Sensitivity (%),Specificity (%),Precision / PPV (%),NPV (%),Accuracy (%),F1-Score,Miss Rate / FNR (%),Fall-Out / FPR (%),FDR (%)
0,Algo 1,965,17784,312,939,50.68,98.28,75.57,94.98,93.75,0.6067,49.32,1.72,24.43
1,Algo 2,1743,7749,10347,161,91.54,42.82,14.42,97.96,47.46,0.2491,8.46,57.18,85.58
2,Algo 3,1403,17559,537,501,73.69,97.03,72.32,97.23,94.81,0.7300,26.31,2.97,27.68


In [9]:
age_df = compute_age_subgroup_df(filtered_df)
display(age_df)
age_subgroup_chart(age_df).show()
ag_fine_df = compute_age_gender_fine_df(filtered_df)
display(ag_fine_df.head())
age_gender_sensitivity_line_chart(ag_fine_df, filtered_df).show()
age_gender_metric_chart(ag_fine_df, "Spec", "Specificity (%)", "Specificity by Age Group and Gender").show()
site_acc = site_accuracy_df(filtered_df)
display(site_acc)
site_long = site_acc.melt(id_vars="site", value_vars=ALGO_COLS, var_name="Algorithm", value_name="Accuracy (%)")
fig_site = px.bar(site_long, x="site", y="Accuracy (%)", color="Algorithm", barmode="group", title="Accuracy (%) by Hospital Site", color_discrete_map=COLOR_MAP, text=site_long["Accuracy (%)"].map(lambda v: f"{v:.1f}%"))
fig_site.update_traces(textposition="outside")
fig_site.update_layout(legend_title_text="")
soften_bar_figure(fig_site).show()

,age_group,scans,Radiologist,Algo 1,Algo 2,Algo 3
0,05-08,50,4.0,94.0,54.0,98.0
1,08-11,156,10.3,92.3,66.0,94.9
2,11-14,251,11.6,92.8,62.5,94.0
3,14-17,362,12.2,93.4,64.6,93.4
4,17-20,392,12.8,94.1,60.7,94.9
5,20-23,486,14.2,91.2,62.1,91.6
6,23-26,633,9.5,93.0,60.7,95.7
7,26-29,698,10.5,93.4,60.5,95.4
8,29-32,834,10.8,93.6,62.4,93.2
9,32-35,957,13.0,92.4,61.7,94.0


<string>:327: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


,age_group,gender,scans,Algo 1 Sens,Algo 1 Spec,Algo 2 Sens,Algo 2 Spec,Algo 3 Sens,Algo 3 Spec
52,05-08,female,20,0.0,100.0,100.0,47.4,100.0,100.0
53,05-08,male,30,0.0,96.6,0.0,58.6,0.0,100.0
49,08-11,female,66,25.0,98.3,87.5,60.3,62.5,98.3
45,08-11,male,90,62.5,97.6,100.0,64.6,100.0,95.1
50,11-14,female,125,58.3,97.3,91.7,57.5,83.3,94.7


<string>:381: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
<string>:381: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
<string>:381: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select t

,site,scans,Algo 1,Algo 2,Algo 3
0,best_doctors,5986,93.8,47.6,94.7
1,healthy_vibes,14014,93.7,47.4,94.9


## Exploratory — Algorithm Agreement

In [10]:
gt_pos = filtered_df["radiologist_answer"] == "P"
fn1 = gt_pos & (filtered_df["algo1_answer"] == "N")
fn2 = gt_pos & (filtered_df["algo2_answer"] == "N")
fn3 = gt_pos & (filtered_df["algo3_answer"] == "N")
all_fn = fn1 & fn2 & fn3

fn_summary = pd.DataFrame([
    {"Case set": "False negative — Algo 1", "Count": int(fn1.sum())},
    {"Case set": "False negative — Algo 2", "Count": int(fn2.sum())},
    {"Case set": "False negative — Algo 3", "Count": int(fn3.sum())},
    {"Case set": "False negative — intersection (all 3 algos)", "Count": int(all_fn.sum())},
])
display(fn_summary)

cols = [
    "accession", "site", "patient_class", "gender", "age",
    "radiologist_answer", "algo1_answer", "algo2_answer", "algo3_answer",
]
if all_fn.any():
    fn_cases = filtered_df.loc[all_fn, cols].sort_values("accession")
    display(fn_cases)
    print(f"\nAll algos N, radiologist P — accessions ({len(fn_cases)}):")
    print(fn_cases["accession"].tolist())
else:
    print("No scans where all three algos share the same false negative.")


,Case set,Count
0,False negative — Algo 1,939
1,False negative — Algo 2,161
2,False negative — Algo 3,501
3,False negative — intersection (all 3 algos),108


,accession,site,patient_class,gender,age,radiologist_answer,algo1_answer,algo2_answer,algo3_answer
12176,226986,healthy_vibes,IN,female,82,P,N,N,N
12119,883122,healthy_vibes,IN,male,61,P,N,N,N
12244,1161228,best_doctors,IN,female,84,P,N,N,N
12206,1223638,best_doctors,IN,female,54,P,N,N,N
12167,1499153,healthy_vibes,IN,male,65,P,N,N,N
...,...,...,...,...,...,...,...,...,...
12248,96545024,best_doctors,IN,female,51,P,N,N,N
12091,96599991,healthy_vibes,ED,female,36,P,N,N,N
12135,98664812,healthy_vibes,IN,female,56,P,N,N,N
12126,99662162,healthy_vibes,IN,female,46,P,N,N,N



All algos N, radiologist P — accessions (108):
[226986, 883122, 1161228, 1223638, 1499153, 1665522, 4929613, 7860062, 8092873, 8961408, 9374356, 10309337, 10702795, 11215771, 11679970, 13959158, 14200167, 15338848, 15511004, 18132385, 19987627, 20548316, 22741454, 23101246, 23972022, 25841023, 25943500, 25979509, 30908728, 31713660, 32844187, 35303729, 35328120, 35508403, 35550647, 36646819, 36666611, 36900754, 38071787, 38609464, 39015389, 39712353, 40991997, 41834112, 41983180, 42138017, 42212696, 42865662, 44441486, 44747208, 47430012, 48152407, 51235924, 51486694, 51530230, 53670079, 54091744, 55068636, 55448082, 55843740, 56560541, 56671597, 56727913, 57687396, 58317732, 59429424, 61008479, 61313755, 61507451, 61896236, 62839931, 63799962, 64546134, 65903997, 68168667, 68887974, 71058210, 71213608, 73112141, 74189692, 75085966, 77657581, 78601354, 80268136, 80682842, 82690839, 82938385, 85663566, 86037707, 86569974, 86789311, 87387011, 87818408, 87936974, 88576455, 89777118, 9137

In [11]:
unanimous_vs_rad = (
    (filtered_df["algo1_answer"] == filtered_df["algo2_answer"])
    & (filtered_df["algo2_answer"] == filtered_df["algo3_answer"])
    & (filtered_df["algo1_answer"] != filtered_df["radiologist_answer"])
)
all_p_rad_n = unanimous_vs_rad & (filtered_df["algo1_answer"] == "P")
all_n_rad_p = unanimous_vs_rad & (filtered_df["algo1_answer"] == "N")

summary = pd.DataFrame([
    {"Pattern": "All algos P, radiologist N", "Count": int(all_p_rad_n.sum())},
    {"Pattern": "All algos N, radiologist P", "Count": int(all_n_rad_p.sum())},
    {"Pattern": "Any unanimous algo vs radiologist", "Count": int(unanimous_vs_rad.sum())},
])
display(summary)

cols = [
    "accession", "site", "patient_class", "gender", "age",
    "radiologist_answer", "algo1_answer", "algo2_answer", "algo3_answer",
]

for label, mask in [
    ("All algos P, radiologist N", all_p_rad_n),
    ("All algos N, radiologist P", all_n_rad_p),
]:
    cases = filtered_df.loc[mask, cols].assign(pattern=label).sort_values("accession")
    if cases.empty:
        print(f"\n{label}: none")
        continue
    display(cases)
    print(f"\n{label} — accessions ({len(cases)}):")
    print(cases["accession"].tolist())

all_cases = filtered_df.loc[unanimous_vs_rad, cols].copy()
all_cases["pattern"] = np.where(all_cases["algo1_answer"] == "P", "All P / Rad N", "All N / Rad P")
all_cases = all_cases.sort_values("accession")
print(f"\nCombined — all unanimous disagreements ({len(all_cases)} accessions):")
print(all_cases["accession"].tolist())


,Pattern,Count
0,"All algos P, radiologist N",3
1,"All algos N, radiologist P",108
2,Any unanimous algo vs radiologist,111


,accession,site,patient_class,gender,age,radiologist_answer,algo1_answer,algo2_answer,algo3_answer,pattern
9520,22927914,best_doctors,ED,male,72,N,P,P,P,"All algos P, radiologist N"
6680,34059032,healthy_vibes,IN,female,63,N,P,P,P,"All algos P, radiologist N"
10387,77771253,best_doctors,IN,male,71,N,P,P,P,"All algos P, radiologist N"



All algos P, radiologist N — accessions (3):
[22927914, 34059032, 77771253]


,accession,site,patient_class,gender,age,radiologist_answer,algo1_answer,algo2_answer,algo3_answer,pattern
12176,226986,healthy_vibes,IN,female,82,P,N,N,N,"All algos N, radiologist P"
12119,883122,healthy_vibes,IN,male,61,P,N,N,N,"All algos N, radiologist P"
12244,1161228,best_doctors,IN,female,84,P,N,N,N,"All algos N, radiologist P"
12206,1223638,best_doctors,IN,female,54,P,N,N,N,"All algos N, radiologist P"
12167,1499153,healthy_vibes,IN,male,65,P,N,N,N,"All algos N, radiologist P"
...,...,...,...,...,...,...,...,...,...,...
12248,96545024,best_doctors,IN,female,51,P,N,N,N,"All algos N, radiologist P"
12091,96599991,healthy_vibes,ED,female,36,P,N,N,N,"All algos N, radiologist P"
12135,98664812,healthy_vibes,IN,female,56,P,N,N,N,"All algos N, radiologist P"
12126,99662162,healthy_vibes,IN,female,46,P,N,N,N,"All algos N, radiologist P"



All algos N, radiologist P — accessions (108):
[226986, 883122, 1161228, 1223638, 1499153, 1665522, 4929613, 7860062, 8092873, 8961408, 9374356, 10309337, 10702795, 11215771, 11679970, 13959158, 14200167, 15338848, 15511004, 18132385, 19987627, 20548316, 22741454, 23101246, 23972022, 25841023, 25943500, 25979509, 30908728, 31713660, 32844187, 35303729, 35328120, 35508403, 35550647, 36646819, 36666611, 36900754, 38071787, 38609464, 39015389, 39712353, 40991997, 41834112, 41983180, 42138017, 42212696, 42865662, 44441486, 44747208, 47430012, 48152407, 51235924, 51486694, 51530230, 53670079, 54091744, 55068636, 55448082, 55843740, 56560541, 56671597, 56727913, 57687396, 58317732, 59429424, 61008479, 61313755, 61507451, 61896236, 62839931, 63799962, 64546134, 65903997, 68168667, 68887974, 71058210, 71213608, 73112141, 74189692, 75085966, 77657581, 78601354, 80268136, 80682842, 82690839, 82938385, 85663566, 86037707, 86569974, 86789311, 87387011, 87818408, 87936974, 88576455, 89777118, 9137

## Correlations

Do longer radiologist turnaround times go together with longer algorithm processing on the same scan?


In [12]:
mask = (
    filtered_df["scan_timestamp"].notna()
    & filtered_df["radiologist_sign_time"].notna()
    & filtered_df["algos_start_run"].notna()
)
sub = filtered_df.loc[mask].copy()
sub["hour"] = sub["scan_timestamp"].dt.hour.astype(int)
sub["rad_min"] = (sub["radiologist_sign_time"] - sub["algos_start_run"]).dt.total_seconds() / 60.0
sub = sub[sub["rad_min"] >= 0]
hourly_overall_df = (
    sub.groupby("hour", as_index=False)
    .agg(
        scans=("radiologist_answer", "count"),
        pos_rate=("radiologist_answer", lambda s: round((s == "P").mean() * 100, 1)),
        mean_rad_min=("rad_min", lambda s: round(s.mean(), 2)),
    )
    .sort_values("hour")
)
display(hourly_overall_df)

fig_corr = px.scatter(
    hourly_overall_df, x="scans", y="mean_rad_min",
    text="hour", size="pos_rate",
    title="Correlation: Scan Volume vs Radiologist Time by Hour",
    labels={"scans": "Scans per Hour", "mean_rad_min": "Mean Rad Time (min)"},
)
fig_corr.update_traces(textposition="top center")
corr = float(np.corrcoef(hourly_overall_df["scans"], hourly_overall_df["mean_rad_min"])[0, 1])
slope, intercept = np.polyfit(hourly_overall_df["scans"], hourly_overall_df["mean_rad_min"], 1)
x_range = [hourly_overall_df["scans"].min(), hourly_overall_df["scans"].max()]
fig_corr.add_trace(go.Scatter(
    x=x_range, y=[slope * x + intercept for x in x_range],
    mode="lines", name=f"Trend (r={corr:.3f})", line=dict(color="grey", dash="dash"),
))
fig_corr.update_layout(showlegend=True)
fig_corr.show()
print(f"Pearson r = {corr:.4f}")

,hour,scans,pos_rate,mean_rad_min
0,0,872,11.1,9.73
1,1,837,11.2,9.71
2,2,830,11.3,9.76
3,3,868,10.5,9.77
4,4,816,8.7,9.75
5,5,807,7.7,9.65
6,6,815,8.1,9.68
7,7,845,10.4,9.77
8,8,829,8.3,9.75
9,9,854,10.8,9.64


Pearson r = 0.1590


In [13]:
duration_df = compute_duration_df(filtered_df)
print(f"Valid TAT rows: {len(duration_df):,}")

rad = duration_df["Radiologist"]
corr_rows = []
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"{algo} vs Radiologist" for algo in ALGO_COLS],
    horizontal_spacing=0.08,
)

for col_idx, algo in enumerate(ALGO_COLS, start=1):
    algo_tat = duration_df[algo]
    r = float(np.corrcoef(rad, algo_tat)[0, 1])
    slope, intercept = np.polyfit(rad, algo_tat, 1)
    corr_rows.append({"Comparison": f"Radiologist vs {algo}", "Pearson r": round(r, 4)})

    fig.add_trace(
        go.Scatter(
            x=rad, y=algo_tat, mode="markers",
            marker=dict(size=3, color=COLOR_MAP[algo], opacity=0.2),
            name=algo, showlegend=False,
            hovertemplate=(
                f"Radiologist: %{{x:.1f}} min<br>{algo}: %{{y:.1f}} min<extra></extra>"
            ),
        ),
        row=1, col=col_idx,
    )
    x_range = np.linspace(rad.min(), rad.max(), 2)
    fig.add_trace(
        go.Scatter(
            x=x_range, y=slope * x_range + intercept, mode="lines",
            line=dict(color=COLOR_MAP[algo], dash="dash", width=2),
            showlegend=False, hoverinfo="skip",
        ),
        row=1, col=col_idx,
    )
    fig.layout.annotations[col_idx - 1].text = f"{algo} vs Radiologist<br>r = {r:.3f}"

display(pd.DataFrame(corr_rows))
fig.update_layout(
    title="Radiologist TAT vs Algorithm TAT (per scan)",
    height=420, width=1100,
    margin=dict(t=80, b=50),
)
for col_idx in range(1, 4):
    fig.update_xaxes(title_text="Radiologist (min)", row=1, col=col_idx)
    fig.update_yaxes(title_text=f"{ALGO_COLS[col_idx - 1]} (min)", row=1, col=col_idx)
fig.show()

print(
    "Interpretation: r near 0 → algorithm time is largely independent of how long "
    "the radiologist took on that scan; higher r → both tend to be slow/fast together."
)


Valid TAT rows: 19,982


,Comparison,Pearson r
0,Radiologist vs Algo 1,0.0062
1,Radiologist vs Algo 2,0.0087
2,Radiologist vs Algo 3,0.0093


Interpretation: r near 0 → algorithm time is largely independent of how long the radiologist took on that scan; higher r → both tend to be slow/fast together.


---
## Appendix — Helper definitions

**No more queries below.** Chart/query helpers used by the sections above.

In [14]:
COLOR_MAP = {
    "Radiologist": "#2ca02c",
    "Algo 1": "#d62728",
    "Algo 2": "#1f77b4",
    "Algo 3": "#ff7f0e",
}
ALGO_COLS = ["Algo 1", "Algo 2", "Algo 3"]
AGE_LINE_COLS = ["Radiologist", "Algo 1", "Algo 2", "Algo 3"]
ALGO_ANSWER_COLS = ["algo1_answer", "algo2_answer", "algo3_answer"]

AGE_GROUP_ORDER = ["00-17", "18-39", "40-64", "65+"]
PATIENT_CLASS_ORDER = ["ED", "IN"]
GENDER_COLOR_MAP = {"male": "#2a9d8f", "female": "#bc4749"}
DEPT_COLOR_MAP = {"ED": "#bc4749", "IN": "#2a9d8f"}
DEPT_ALGO_COLORS = {
    "ED": {"Algo 1": "#d62728", "Algo 2": "#1f77b4", "Algo 3": "#ff7f0e"},
    "IN": {"Algo 1": "#e07a7f", "Algo 2": "#6baed6", "Algo 3": "#fdb462"},
}
DEPT_LINE_DASH = {"ED": "solid", "IN": "dash"}
BAR_FILL_OPACITY = 0.72


def soften_bar_figure(fig: go.Figure, opacity: float = BAR_FILL_OPACITY) -> go.Figure:
    for trace in fig.data:
        if trace.type == "bar":
            trace.update(marker=dict(opacity=opacity), opacity=opacity)
    return fig

FINE_AGE_EDGES = [
    5, 8, 11, 14, 17, 20, 23, 26, 29, 32, 35, 38, 41, 44, 47, 50, 53, 56,
    59, 62, 65, 68, 71, 74, 77, 80, 83, np.inf,
]
FINE_AGE_LABELS = [
    "05-08", "08-11", "11-14", "14-17", "17-20", "20-23", "23-26", "26-29",
    "29-32", "32-35", "35-38", "38-41", "41-44", "44-47", "47-50", "50-53",
    "53-56", "56-59", "59-62", "62-65", "65-68", "68-71", "71-74", "74-77",
    "77-80", "80-83", "83+",
]

MONTH_LABELS = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec",
}
DOW_LABELS = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}


def assign_age_group_series(ages: pd.Series) -> pd.Series:
    return pd.Series(
        np.select(
            [ages < 18, ages < 40, ages < 65],
            ["00-17", "18-39", "40-64"],
            default="65+",
        ),
        index=ages.index,
    )


def filter_dataframe(
    df: pd.DataFrame,
    sites: list,
    patient_classes: list,
    genders: list,
    age_lo: int,
    age_hi: int,
) -> pd.DataFrame:
    return df[
        df["site"].isin(sites)
        & df["patient_class"].isin(patient_classes)
        & df["gender"].isin(genders)
        & (df["age"] >= age_lo)
        & (df["age"] <= age_hi)
    ].copy()


def group_distribution_df(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    out = (
        df.groupby(group_col, as_index=False)
        .agg(
            scans=("radiologist_answer", "count"),
            positives=("radiologist_answer", lambda s: (s == "P").sum()),
        )
        .assign(
            prevalence_pct=lambda d: (d["positives"] / d["scans"] * 100).round(1)
        )
        .sort_values(group_col)
    )
    return out


def group_accuracy_df(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    out = group_distribution_df(df, group_col)
    for name, col in zip(ALGO_COLS, ALGO_ANSWER_COLS):
        out[name] = (
            df.groupby(group_col)
            .apply(lambda g, c=col: (g[c] == g["radiologist_answer"]).mean() * 100)
            .round(1)
            .values
        )
    return out


def volume_sunburst_df(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby(["site", "patient_class", "gender"], as_index=False)
        .agg(
            scans=("radiologist_answer", "count"),
            positives=("radiologist_answer", lambda s: (s == "P").sum()),
        )
        .assign(
            prevalence_pct=lambda d: (d["positives"] / d["scans"] * 100).round(1)
        )
        .sort_values(["site", "patient_class", "gender"])
    )


def age_prevalence_sunburst_df(df: pd.DataFrame) -> pd.DataFrame:
    binned = df.assign(age_group=assign_age_group_series(df["age"]))
    out = (
        binned.groupby(["patient_class", "age_group"], as_index=False)
        .agg(
            scans=("radiologist_answer", "count"),
            positives=("radiologist_answer", lambda s: (s == "P").sum()),
        )
        .assign(
            prevalence_pct=lambda d: (d["positives"] / d["scans"] * 100).round(1)
        )
    )
    out["patient_class"] = pd.Categorical(
        out["patient_class"], categories=PATIENT_CLASS_ORDER, ordered=True
    )
    out["age_group"] = pd.Categorical(
        out["age_group"], categories=AGE_GROUP_ORDER, ordered=True
    )
    return out.sort_values(["patient_class", "age_group"])


def build_temporal_base(df: pd.DataFrame) -> pd.DataFrame:
    base = df[df["scan_timestamp"].notna()].copy()
    base["hour"] = base["scan_timestamp"].dt.hour.astype(int)
    base["dow"] = base["scan_timestamp"].dt.dayofweek.astype(int)
    base["month"] = base["scan_timestamp"].dt.month.astype(int)
    return base


def hourly_total_scans(temporal_base: pd.DataFrame) -> pd.DataFrame:
    return (
        temporal_base.groupby("hour", as_index=False)
        .agg(scans=("radiologist_answer", "count"))
        .sort_values("hour")
    )


def hourly_pos_rate(temporal_base: pd.DataFrame) -> pd.DataFrame:
    return (
        temporal_base.groupby("hour", as_index=False)
        .agg(pos_rate=("radiologist_answer", lambda s: round((s == "P").mean() * 100, 1)))
        .sort_values("hour")
    )


def monthly_pos_rate(temporal_base: pd.DataFrame) -> pd.DataFrame:
    out = (
        temporal_base.groupby("month", as_index=False)
        .agg(pos_rate=("radiologist_answer", lambda s: round((s == "P").mean() * 100, 1)))
        .sort_values("month")
    )
    out["month_name"] = out["month"].map(MONTH_LABELS)
    return out


def heatmap_data(temporal_base: pd.DataFrame) -> pd.DataFrame:
    return (
        temporal_base.groupby(["dow", "hour"], as_index=False)
        .agg(
            scans=("radiologist_answer", "count"),
            pos_rate=("radiologist_answer", lambda s: round((s == "P").mean() * 100, 1)),
        )
        .sort_values(["dow", "hour"])
    )


def count_with_times(df: pd.DataFrame) -> int:
    required = [
        "radiologist_sign_time",
        "algos_start_run",
        "algo1_finish_run",
        "algo2_finish_run",
        "algo3_finish_run",
    ]
    return int(df[required].notna().all(axis=1).sum())


def compute_duration_df(df: pd.DataFrame) -> pd.DataFrame:
    required = [
        "radiologist_sign_time",
        "algos_start_run",
        "algo1_finish_run",
        "algo2_finish_run",
        "algo3_finish_run",
    ]
    sub = df[df[required].notna().all(axis=1)].copy()
    duration_df = pd.DataFrame(
        {
            "Radiologist": (
                sub["radiologist_sign_time"] - sub["algos_start_run"]
            ).dt.total_seconds()
            / 60.0,
            "Algo 1": (
                sub["algo1_finish_run"] - sub["algos_start_run"]
            ).dt.total_seconds()
            / 60.0,
            "Algo 2": (
                sub["algo2_finish_run"] - sub["algos_start_run"]
            ).dt.total_seconds()
            / 60.0,
            "Algo 3": (
                sub["algo3_finish_run"] - sub["algos_start_run"]
            ).dt.total_seconds()
            / 60.0,
        }
    )
    valid = (duration_df >= 0).all(axis=1)
    return duration_df.loc[valid]


def hourly_scans_rad_by_group(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    mask = (
        df["scan_timestamp"].notna()
        & df["radiologist_sign_time"].notna()
        & df["algos_start_run"].notna()
    )
    sub = df.loc[mask].copy()
    sub["hour"] = sub["scan_timestamp"].dt.hour.astype(int)
    sub["rad_min"] = (
        sub["radiologist_sign_time"] - sub["algos_start_run"]
    ).dt.total_seconds() / 60.0
    sub = sub[sub["rad_min"] >= 0]
    return (
        sub.groupby(["hour", group_col], as_index=False)
        .agg(
            scans=("radiologist_answer", "count"),
            mean_rad_min=("rad_min", lambda s: round(s.mean(), 2)),
        )
        .sort_values(["hour", group_col])
    )


def hourly_scans_algo_by_group(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    required = [
        "scan_timestamp",
        "algos_start_run",
        "algo1_finish_run",
        "algo2_finish_run",
        "algo3_finish_run",
    ]
    sub = df[df[required].notna().all(axis=1)].copy()
    sub["hour"] = sub["scan_timestamp"].dt.hour.astype(int)
    for i, col in enumerate(ALGO_ANSWER_COLS, 1):
        finish_col = f"algo{i}_finish_run"
        sub[f"algo{i}_min"] = (
            sub[finish_col] - sub["algos_start_run"]
        ).dt.total_seconds() / 60.0
    valid = (sub[[f"algo{i}_min" for i in range(1, 4)]] >= 0).all(axis=1)
    sub = sub[valid]
    return (
        sub.groupby(["hour", group_col], as_index=False)
        .agg(
            scans=("radiologist_answer", "count"),
            **{
                f"mean_{name.lower().replace(' ', '')}_min": (
                    f"algo{i}_min",
                    lambda s, n=name: round(s.mean(), 2),
                )
                for i, name in enumerate(ALGO_COLS, 1)
            },
        )
        .sort_values(["hour", group_col])
    )


def compute_metrics_df(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for name, col in zip(ALGO_COLS, ALGO_ANSWER_COLS):
        tp = int(((df[col] == "P") & (df["radiologist_answer"] == "P")).sum())
        tn = int(((df[col] == "N") & (df["radiologist_answer"] == "N")).sum())
        fp = int(((df[col] == "P") & (df["radiologist_answer"] == "N")).sum())
        fn = int(((df[col] == "N") & (df["radiologist_answer"] == "P")).sum())
        n = tp + tn + fp + fn
        f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan
        npv = tn / (tn + fn) if (tn + fn) else np.nan
        miss_rate = fn / (fn + tp) if (fn + tp) else np.nan
        fall_out = fp / (fp + tn) if (fp + tn) else np.nan
        fdr = fp / (fp + tp) if (fp + tp) else np.nan
        rows.append(
            {
                "Algorithm": name,
                "TP": tp,
                "TN": tn,
                "FP": fp,
                "FN": fn,
                "Sensitivity (%)": round(tp / (tp + fn) * 100, 2) if (tp + fn) else np.nan,
                "Specificity (%)": round(tn / (tn + fp) * 100, 2) if (tn + fp) else np.nan,
                "Precision / PPV (%)": round(tp / (tp + fp) * 100, 2) if (tp + fp) else np.nan,
                "NPV (%)": round(npv * 100, 2) if pd.notna(npv) else np.nan,
                "Accuracy (%)": round((tp + tn) / n * 100, 2) if n else np.nan,
                "F1-Score": round(f1, 4) if pd.notna(f1) else np.nan,
                "Miss Rate / FNR (%)": round(miss_rate * 100, 2) if pd.notna(miss_rate) else np.nan,
                "Fall-Out / FPR (%)": round(fall_out * 100, 2) if pd.notna(fall_out) else np.nan,
                "FDR (%)": round(fdr * 100, 2) if pd.notna(fdr) else np.nan,
            }
        )
    return pd.DataFrame(rows)


def compute_age_gender_fine_df(df: pd.DataFrame) -> pd.DataFrame:
    sub = df[df["age"] >= 5].copy()
    binned = sub.assign(
        age_group=pd.cut(
            sub["age"],
            bins=FINE_AGE_EDGES,
            labels=FINE_AGE_LABELS,
            right=False,
        )
    )
    binned = binned[binned["age_group"].notna()].copy()
    rows = []
    for (age_group, gender), grp in binned.groupby(["age_group", "gender"], sort=False):
        scans = len(grp)
        row = {"age_group": str(age_group), "gender": gender, "scans": scans}
        gt_pos = grp[grp["radiologist_answer"] == "P"]
        gt_neg = grp[grp["radiologist_answer"] == "N"]
        for name, col in zip(ALGO_COLS, ALGO_ANSWER_COLS):
            row[f"{name} Sens"] = (
                round((gt_pos[col] == "P").mean() * 100, 1) if len(gt_pos) else np.nan
            )
            row[f"{name} Spec"] = (
                round((gt_neg[col] == "N").mean() * 100, 1) if len(gt_neg) else np.nan
            )
        rows.append(row)
    out = pd.DataFrame(rows)
    out["age_group"] = pd.Categorical(
        out["age_group"], categories=FINE_AGE_LABELS, ordered=True
    )
    return out.sort_values(["age_group", "gender"])


def compute_age_gender_df(df: pd.DataFrame) -> pd.DataFrame:
    binned = df.assign(age_group=assign_age_group_series(df["age"]))
    rows = []
    for (age_group, gender), grp in binned.groupby(["age_group", "gender"], sort=False):
        scans = len(grp)
        positives = int((grp["radiologist_answer"] == "P").sum())
        gt_pos = grp[grp["radiologist_answer"] == "P"]
        row = {
            "age_group": age_group,
            "gender": gender,
            "scans": scans,
            "positives": positives,
            "prevalence_pct": round(positives / scans * 100, 1),
        }
        for name, col in zip(ALGO_COLS, ALGO_ANSWER_COLS):
            row[f"{name} Acc"] = round((grp[col] == grp["radiologist_answer"]).mean() * 100, 1)
            row[f"{name} Sens"] = (
                round((gt_pos[col] == "P").mean() * 100, 1) if len(gt_pos) else np.nan
            )
        rows.append(row)
    out = pd.DataFrame(rows)
    out["age_group"] = pd.Categorical(
        out["age_group"], categories=AGE_GROUP_ORDER, ordered=True
    )
    return out.sort_values(["age_group", "gender"])


def site_accuracy_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.groupby("site", as_index=False).agg(
        scans=("radiologist_answer", "count")
    )
    for name, col in zip(ALGO_COLS, ALGO_ANSWER_COLS):
        out[name] = (
            df.groupby("site")
            .apply(lambda g, c=col: (g[c] == g["radiologist_answer"]).mean() * 100)
            .round(1)
            .values
        )
    return out.sort_values("site")


def compute_age_subgroup_df(df: pd.DataFrame) -> pd.DataFrame:
    sub = df[df["age"] >= 5].copy()
    binned = sub.assign(
        age_group=pd.cut(
            sub["age"],
            bins=FINE_AGE_EDGES,
            labels=FINE_AGE_LABELS,
            right=False,
        )
    )
    binned = binned[binned["age_group"].notna()].copy()
    for i, col in enumerate(ALGO_ANSWER_COLS, 1):
        binned[f"a{i}_corr"] = (
            binned[col] == binned["radiologist_answer"]
        ).astype(int)
    out = (
        binned.groupby("age_group", as_index=False, observed=False)
        .agg(
            scans=("radiologist_answer", "count"),
            Radiologist=(
                "radiologist_answer",
                lambda s: round((s == "P").mean() * 100, 1),
            ),
            **{
                name: (f"a{i}_corr", lambda s: round(s.mean() * 100, 1))
                for i, name in enumerate(ALGO_COLS, 1)
            },
        )
        .sort_values("age_group")
    )
    return out


def prevalence_sunburst(
    df: pd.DataFrame,
    path: list[str],
    title: str,
    sort_slices: bool = True,
    *,
    compact: bool = False,
    show_title: bool = True,
) -> go.Figure:
    fig = px.sunburst(
        df,
        path=path,
        values="scans",
        title=title if show_title else None,
        color="prevalence_pct",
        color_continuous_scale="Tealrose",
        custom_data=["scans", "positives", "prevalence_pct"],
    )
    fig.update_traces(
        sort=sort_slices,
        textinfo="label+percent parent+value",
        insidetextorientation="radial",
        textfont=dict(size=14 if compact else 15),
        hovertemplate=(
            "<b>%{label}</b><br>Scans: %{customdata[0]:,}"
            "<br>Positives: %{customdata[1]:,}"
            "<br>Prevalence: %{customdata[2]}%<extra></extra>"
        ),
    )
    layout_kwargs = dict(
        height=580 if compact else 700,
        margin=dict(t=20 if compact and not show_title else 60, l=10, r=10, b=10),
        uniformtext_minsize=12,
        coloraxis_colorbar=dict(title="Prevalence %", thickness=12, len=0.75),
    )
    if show_title:
        layout_kwargs["title_font"] = dict(size=18)
    if not compact:
        layout_kwargs["width"] = 900
    fig.update_layout(**layout_kwargs)
    return fig


def age_distribution_by_gender_chart(df: pd.DataFrame, bin_width: int = 5) -> go.Figure:
    ages = df["age"].dropna()
    fig = go.Figure()
    if ages.empty:
        fig.update_layout(title="Age Distribution by Gender")
        return fig

    min_edge = int(np.floor(ages.min() / bin_width) * bin_width)
    max_edge = int(np.ceil(ages.max() / bin_width) * bin_width)
    bin_edges = np.arange(min_edge, max_edge + bin_width, bin_width)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    for gender in ["male", "female"]:
        counts, _ = np.histogram(
            df.loc[df["gender"] == gender, "age"].dropna(),
            bins=bin_edges,
        )
        color = GENDER_COLOR_MAP[gender]
        fig.add_trace(
            go.Bar(
                x=bin_centers,
                y=counts,
                name=gender,
                marker=dict(color=color, line=dict(width=0)),
                opacity=0.72,
                width=bin_width * 0.92,
                text=[str(c) if c else "" for c in counts],
                textposition="outside",
                textfont=dict(color=color, size=11, family="Arial Black, Arial, sans-serif"),
                hovertemplate=(
                    f"{gender}<br>Age %{{customdata}}<br>Count: %{{y:,}}<extra></extra>"
                ),
                customdata=[
                    f"{int(lo)}–{int(hi)}"
                    for lo, hi in zip(bin_edges[:-1], bin_edges[1:])
                ],
            )
        )

    fig.update_layout(
        title="Age Distribution by Gender",
        barmode="overlay",
        xaxis=dict(title="Age (years)", tickmode="linear", dtick=bin_width),
        yaxis=dict(title="Count"),
        legend_title_text="",
        bargap=0.05,
        uniformtext_minsize=9,
        margin=dict(t=70, r=20, b=50, l=50),
    )
    return fig


def tat_dot_plot(duration_df: pd.DataFrame) -> go.Figure:
    entity_order = ["Radiologist", "Algo 1", "Algo 2", "Algo 3"]
    fig = go.Figure()
    rng = np.random.default_rng(42)

    for idx, entity in enumerate(entity_order):
        values = duration_df[entity].dropna().to_numpy()
        mean_val = float(np.mean(values))
        x_jitter = idx + rng.uniform(-0.35, 0.35, len(values))
        color = COLOR_MAP[entity]

        fig.add_trace(
            go.Scatter(
                x=x_jitter,
                y=values,
                mode="markers",
                name=entity,
                marker=dict(color=color, size=4, opacity=0.2),
                hovertemplate=f"{entity}<br>TAT: %{{y:.1f}} min<extra></extra>",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=[idx - 0.38, idx + 0.38],
                y=[mean_val, mean_val],
                mode="lines",
                line=dict(color=color, width=2.5, dash="dash"),
                showlegend=False,
                hoverinfo="skip",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=[idx],
                y=[mean_val],
                mode="markers+text",
                marker=dict(symbol="diamond", size=11, color=color, line=dict(width=1, color="white")),
                text=[f"μ={mean_val:.1f}"],
                textposition="top center",
                textfont=dict(color=color, size=11),
                showlegend=False,
                hovertemplate=f"{entity}<br>Mean: {mean_val:.2f} min<extra></extra>",
            )
        )

    fig.update_layout(
        title="TAT Dot Plot (minutes)",
        xaxis=dict(
            tickvals=list(range(len(entity_order))),
            ticktext=entity_order,
            title="",
        ),
        yaxis_title="Minutes",
        showlegend=False,
        margin=dict(t=50, b=40, l=50, r=20),
    )
    return fig


def tat_win_count_chart(duration_df: pd.DataFrame) -> go.Figure:
    pairs = [
        ("Radiologist", "Algo 1"),
        ("Radiologist", "Algo 2"),
        ("Radiologist", "Algo 3"),
        ("Algo 1", "Algo 2"),
        ("Algo 1", "Algo 3"),
        ("Algo 2", "Algo 3"),
    ]
    fig = go.Figure()
    for name_a, name_b in pairs:
        a_wins = int((duration_df[name_a] < duration_df[name_b]).sum())
        b_wins = int((duration_df[name_a] > duration_df[name_b]).sum())
        total = len(duration_df)
        matchup = f"{name_a} vs {name_b}"
        fig.add_trace(
            go.Bar(
                y=[matchup], x=[a_wins / total * 100],
                orientation="h",
                marker_color=COLOR_MAP.get(name_a, "#636363"),
                text=[f"{name_a}: {a_wins:,} ({a_wins / total * 100:.1f}%)"],
                textposition="inside",
                showlegend=False,
                hovertemplate=f"{name_a} faster: {a_wins:,} ({a_wins / total * 100:.1f}%)<extra></extra>",
            )
        )
        fig.add_trace(
            go.Bar(
                y=[matchup], x=[b_wins / total * 100],
                orientation="h",
                marker_color=COLOR_MAP.get(name_b, "#636363"),
                text=[f"{name_b}: {b_wins:,} ({b_wins / total * 100:.1f}%)"],
                textposition="inside",
                showlegend=False,
                hovertemplate=f"{name_b} faster: {b_wins:,} ({b_wins / total * 100:.1f}%)<extra></extra>",
            )
        )
    fig.update_layout(
        title="Who Finishes First? (% of scans)",
        barmode="stack",
        xaxis_title="% of scans",
        yaxis_title="",
        height=350,
        margin=dict(l=150, t=50, b=40, r=20),
    )
    return soften_bar_figure(fig)


def tat_clinical_gap_chart(
    duration_df: pd.DataFrame, threshold: float = 0.5
) -> tuple[go.Figure, float, float]:
    diff = duration_df["Radiologist"] - duration_df["Algo 3"]
    mean_gap = float(diff.mean())
    median_gap = float(diff.median())
    rad_slower = int((diff > threshold).sum())
    algo3_slower = int((diff < -threshold).sum())
    within_threshold = len(diff) - rad_slower - algo3_slower

    clinical = pd.DataFrame([
        {
            "Category": f"Radiologist slower by >{threshold} min",
            "Count": rad_slower,
            "Pct": f"{rad_slower / len(diff) * 100:.1f}%",
        },
        {
            "Category": f"Algo 3 slower by >{threshold} min",
            "Count": algo3_slower,
            "Pct": f"{algo3_slower / len(diff) * 100:.1f}%",
        },
        {
            "Category": f"Within ±{threshold} min",
            "Count": within_threshold,
            "Pct": f"{within_threshold / len(diff) * 100:.1f}%",
        },
    ])
    clinical["Label"] = clinical.apply(
        lambda r: f"{int(r['Count']):,}<br>{r['Pct']}", axis=1
    )
    fig = px.bar(
        clinical,
        x="Category",
        y="Count",
        text="Label",
        title=f"Clinical Threshold: TAT Gap > {threshold} min (Radiologist vs Algo 3)",
        color="Category",
        color_discrete_map={
            f"Radiologist slower by >{threshold} min": COLOR_MAP["Radiologist"],
            f"Algo 3 slower by >{threshold} min": COLOR_MAP["Algo 3"],
            f"Within ±{threshold} min": "rgba(128,128,128,0.5)",
        },
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="Scans")
    return soften_bar_figure(fig), mean_gap, median_gap


def hourly_scans_rad_chart(
    df: pd.DataFrame,
    group_col: str,
    color_map: dict[str, str],
    title: str,
) -> go.Figure:
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for group, color in color_map.items():
        sub = df[df[group_col] == group]
        fig.add_trace(
            go.Bar(
                x=sub["hour"], y=sub["scans"],
                name=f"{group} scans",
                marker_color=color, opacity=BAR_FILL_OPACITY,
                text=[f"{int(v):,}" for v in sub["scans"]],
                textposition="outside",
                textfont=dict(size=9),
            ),
            secondary_y=False,
        )
        fig.add_trace(
            go.Scatter(
                x=sub["hour"], y=sub["mean_rad_min"],
                mode="lines+markers",
                name=f"{group} rad time",
                line=dict(color=color, dash="dot", width=2),
                marker=dict(color=color, size=6),
            ),
            secondary_y=True,
        )
    fig.update_layout(
        title=title,
        barmode="group",
        legend_title_text="",
        margin=dict(t=60, b=50, l=50, r=50),
    )
    fig.update_yaxes(title_text="Total Scans", secondary_y=False)
    fig.update_yaxes(title_text="Mean Rad Time (min)", secondary_y=True)
    fig.update_xaxes(title_text="Hour (0-23)")
    return fig


def hourly_scans_algo_chart(
    df: pd.DataFrame,
    group_col: str,
) -> go.Figure:
    algo_col_map = {
        "Algo 1": "mean_algo1_min",
        "Algo 2": "mean_algo2_min",
        "Algo 3": "mean_algo3_min",
    }
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for dept in PATIENT_CLASS_ORDER:
        dept_color = DEPT_COLOR_MAP[dept]
        sub = df[df[group_col] == dept]
        fig.add_trace(
            go.Bar(
                x=sub["hour"],
                y=sub["scans"],
                name=f"{dept} scans",
                marker_color=dept_color,
                opacity=BAR_FILL_OPACITY,
                text=[f"{int(v):,}" for v in sub["scans"]],
                textposition="outside",
                textfont=dict(size=9),
                legendgroup=f"{dept}-bars",
            ),
            secondary_y=False,
        )
        for algo, col in algo_col_map.items():
            algo_color = DEPT_ALGO_COLORS[dept][algo]
            fig.add_trace(
                go.Scatter(
                    x=sub["hour"],
                    y=sub[col],
                    mode="lines+markers",
                    name=f"{dept} — {algo}",
                    line=dict(
                        color=algo_color,
                        dash=DEPT_LINE_DASH[dept],
                        width=2.5 if dept == "ED" else 2,
                    ),
                    marker=dict(color=algo_color, size=5),
                    legendgroup=f"{dept}-{algo}",
                ),
                secondary_y=True,
            )
    fig.update_layout(
        title="Hourly Scan Volume & Algo Time — by Department",
        barmode="group",
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02,
            font=dict(size=10),
            tracegroupgap=4,
        ),
        legend_title_text="",
        margin=dict(t=70, b=50, l=50, r=160),
    )
    fig.update_yaxes(title_text="Total Scans", secondary_y=False)
    fig.update_yaxes(title_text="Mean Algo Time (min)", secondary_y=True)
    fig.update_xaxes(title_text="Hour (0-23)")
    return fig


def diagnostic_radar_chart(metrics_df: pd.DataFrame) -> go.Figure:
    radar_metrics = [
        "Sensitivity (%)",
        "Specificity (%)",
        "Precision / PPV (%)",
        "Accuracy (%)",
    ]
    fig = go.Figure()
    for _, row in metrics_df.iterrows():
        values = [row[m] for m in radar_metrics]
        fig.add_trace(
            go.Scatterpolar(
                r=values + [values[0]],
                theta=radar_metrics + [radar_metrics[0]],
                name=row["Algorithm"],
                line=dict(color=COLOR_MAP[row["Algorithm"]]),
                fill="toself",
                fillcolor=COLOR_MAP[row["Algorithm"]],
                opacity=0.45,
            )
        )
    fig.update_layout(
        title="Diagnostic Profile",
        polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
        showlegend=True,
        legend=dict(
            orientation="v",
            yanchor="middle",
            y=0.5,
            xanchor="left",
            x=1.08,
            font=dict(size=10),
        ),
        height=320,
        margin=dict(t=50, b=20, l=40, r=110),
    )
    return fig


def _cm_cell_text(label: str, count: int, total: int) -> str:
    pct = count / total * 100 if total else 0
    return f"{label}<br>{count:,}<br>({pct:.1f}%)"


def confusion_matrices_chart(metrics_df: pd.DataFrame) -> go.Figure:
    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=list(metrics_df["Algorithm"]),
        horizontal_spacing=0.12,
    )
    for i, row in metrics_df.iterrows():
        tp, tn, fp, fn = int(row["TP"]), int(row["TN"]), int(row["FP"]), int(row["FN"])
        total = tp + tn + fp + fn
        cm = [[tn, fp], [fn, tp]]
        text = [
            [_cm_cell_text("TN", tn, total), _cm_cell_text("FP", fp, total)],
            [_cm_cell_text("FN", fn, total), _cm_cell_text("TP", tp, total)],
        ]
        fig.add_trace(
            go.Heatmap(
                z=cm,
                x=["N", "P"],
                y=["N", "P"],
                text=text,
                texttemplate="%{text}",
                colorscale=[[0, "#f0fdf4"], [0.5, "#86efac"], [1, "#15803d"]],
                showscale=(i == len(metrics_df) - 1),
                hovertemplate="%{text}<extra></extra>",
                xgap=2,
                ygap=2,
            ),
            row=1,
            col=i + 1,
        )

    for col in (1, 2, 3):
        fig.update_xaxes(
            title_text="Predicted" if col == 2 else "",
            title_standoff=8,
            row=1,
            col=col,
        )
        fig.update_yaxes(
            title_text="Actual" if col == 1 else "",
            title_standoff=8,
            showticklabels=(col == 1),
            row=1,
            col=col,
        )

    fig.update_layout(
        title="Confusion Matrices",
        height=380,
        margin=dict(t=60, b=55, l=70, r=50),
    )
    return fig


def f1_score_chart(metrics_df: pd.DataFrame) -> go.Figure:
    fig = px.bar(
        metrics_df,
        x="Algorithm",
        y="F1-Score",
        color="Algorithm",
        color_discrete_map=COLOR_MAP,
        title="F1-Score per Algorithm",
        text=metrics_df["F1-Score"].map(lambda v: f"{v:.4f}"),
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(
        showlegend=False,
        yaxis=dict(range=[0, 1], title="F1-Score"),
        xaxis_title="",
        height=350,
        margin=dict(t=50, b=40),
    )
    return soften_bar_figure(fig)


def npv_chart(metrics_df: pd.DataFrame) -> go.Figure:
    max_npv = float(metrics_df["NPV (%)"].max())
    y_max = max_npv + max(4.0, max_npv * 0.04)
    fig = px.bar(
        metrics_df,
        x="Algorithm",
        y="NPV (%)",
        color="Algorithm",
        color_discrete_map=COLOR_MAP,
        title="NPV per Algorithm",
        text=metrics_df["NPV (%)"].map(lambda v: f"{v:.2f}%"),
    )
    fig.update_traces(textposition="outside", cliponaxis=False)
    fig.update_layout(
        showlegend=False,
        yaxis=dict(range=[0, y_max], title="NPV (%)"),
        xaxis_title="",
        height=350,
        margin=dict(t=55, b=40),
    )
    return soften_bar_figure(fig)


def error_rate_chart(metrics_df: pd.DataFrame) -> go.Figure:
    rate_cols = ["Miss Rate / FNR (%)", "Fall-Out / FPR (%)", "FDR (%)"]
    long = metrics_df.melt(
        id_vars="Algorithm",
        value_vars=rate_cols,
        var_name="Metric",
        value_name="Rate (%)",
    )
    fig = px.bar(
        long,
        x="Metric",
        y="Rate (%)",
        color="Algorithm",
        barmode="group",
        color_discrete_map=COLOR_MAP,
        title="Error Rate Metrics (lower = better)",
        text="Rate (%)",
    )
    fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
    fig.update_layout(
        height=400,
        xaxis_title="",
        yaxis_title="Rate (%)",
        legend_title_text="",
        margin=dict(t=60, b=40),
    )
    return soften_bar_figure(fig)


def overall_sensitivity_pct(df: pd.DataFrame, gender: str, algo_col: str) -> float:
    sub = df[(df["gender"] == gender) & (df["age"] >= 5)]
    gt_pos = sub[sub["radiologist_answer"] == "P"]
    if gt_pos.empty:
        return np.nan
    return round((gt_pos[algo_col] == "P").mean() * 100, 1)


def age_gender_sensitivity_line_chart(
    ag_df: pd.DataFrame, source_df: pd.DataFrame
) -> go.Figure:
    x_categories = [str(label) for label in FINE_AGE_LABELS]
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=["Male", "Female"],
        shared_yaxes=True,
    )
    for col_idx, gender in enumerate(["male", "female"], start=1):
        sub = ag_df[ag_df["gender"] == gender].sort_values("age_group")
        for algo in ALGO_COLS:
            col_name = f"{algo} Sens"
            algo_col = ALGO_ANSWER_COLS[ALGO_COLS.index(algo)]
            color = COLOR_MAP[algo]
            mean_sens = overall_sensitivity_pct(source_df, gender, algo_col)

            fig.add_trace(
                go.Scatter(
                    x=sub["age_group"].astype(str),
                    y=sub[col_name],
                    mode="lines",
                    name=algo,
                    line=dict(color=color, width=2.5, shape="linear"),
                    legendgroup=algo,
                    showlegend=(col_idx == 1),
                    connectgaps=False,
                    hovertemplate=(
                        f"{algo}<br>Age %{{x}}<br>Sensitivity: %{{y:.1f}}%<extra></extra>"
                    ),
                ),
                row=1,
                col=col_idx,
            )
            if pd.notna(mean_sens):
                fig.add_trace(
                    go.Scatter(
                        x=x_categories,
                        y=[mean_sens] * len(x_categories),
                        mode="lines",
                        name=f"{algo} mean ({mean_sens:.1f}%)",
                        line=dict(color=color, width=2, dash="dash"),
                        legendgroup=algo,
                        showlegend=(col_idx == 1),
                        hovertemplate=f"{algo} overall mean<br>Sensitivity: {mean_sens:.1f}%<extra></extra>",
                    ),
                    row=1,
                    col=col_idx,
                )
    fig.update_layout(
        title="Sensitivity by Age Group and Gender",
        height=420,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.14,
            xanchor="center",
            x=0.5,
        ),
        margin=dict(t=110, b=80, l=50, r=20),
    )
    fig.update_yaxes(title_text="Sensitivity (%)", range=[0, 100], row=1, col=1)
    fig.update_yaxes(range=[0, 100], row=1, col=2)
    for col_idx in (1, 2):
        fig.update_xaxes(
            title_text="Age group (years)",
            type="category",
            categoryorder="array",
            categoryarray=x_categories,
            tickangle=-45,
            row=1,
            col=col_idx,
        )
    return fig


def age_gender_metric_chart(
    ag_df: pd.DataFrame,
    metric_suffix: str,
    y_label: str,
    title: str,
) -> go.Figure:
    x_categories = [str(label) for label in FINE_AGE_LABELS]
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=["Male", "Female"],
        shared_yaxes=True,
    )
    for col_idx, gender in enumerate(["male", "female"], start=1):
        sub = ag_df[ag_df["gender"] == gender].sort_values("age_group")
        for algo in ALGO_COLS:
            col_name = f"{algo} {metric_suffix}"
            fig.add_trace(
                go.Scatter(
                    x=sub["age_group"].astype(str),
                    y=sub[col_name],
                    mode="lines+markers",
                    name=algo,
                    line=dict(color=COLOR_MAP[algo], width=2.5),
                    marker=dict(color=COLOR_MAP[algo], size=7),
                    legendgroup=algo,
                    showlegend=(col_idx == 1),
                    connectgaps=False,
                ),
                row=1,
                col=col_idx,
            )
    fig.update_layout(
        title=title,
        height=420,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.12,
            xanchor="center",
            x=0.5,
        ),
        margin=dict(t=100, b=80, l=50, r=20),
    )
    fig.update_yaxes(title_text=y_label, range=[0, 100], row=1, col=1)
    fig.update_yaxes(range=[0, 100], row=1, col=2)
    for col_idx in (1, 2):
        fig.update_xaxes(
            title_text="Age group (years)",
            type="category",
            categoryorder="array",
            categoryarray=x_categories,
            tickangle=-45,
            row=1,
            col=col_idx,
        )
    return fig


def accuracy_bar(df: pd.DataFrame, group_col: str, title: str) -> go.Figure:
    long = df.melt(
        id_vars=group_col,
        value_vars=ALGO_COLS,
        var_name="Algorithm",
        value_name="Accuracy (%)",
    )
    fig = px.bar(
        long,
        y=group_col,
        x="Accuracy (%)",
        color="Algorithm",
        barmode="group",
        orientation="h",
        title=title,
        color_discrete_map=COLOR_MAP,
        labels={group_col: group_col.replace("_", " ").title()},
        text=long["Accuracy (%)"].map(lambda v: f"{v:.1f}%"),
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(legend_title_text="", yaxis_title="")
    return soften_bar_figure(fig)


def age_subgroup_chart(age_df: pd.DataFrame) -> go.Figure:
    algo_legend_names = {
        "Radiologist": "Radiologist (prevalence)",
        "Algo 1": "Algo 1 (accuracy)",
        "Algo 2": "Algo 2 (accuracy)",
        "Algo 3": "Algo 3 (accuracy)",
    }
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Bar(
            x=age_df["age_group"],
            y=age_df["scans"],
            name="Scan count",
            marker_color="rgba(128, 128, 128, 0.25)",
            text=[f"{int(v):,}" for v in age_df["scans"]],
            textposition="outside",
            textfont=dict(size=9, color="#666"),
            hovertemplate="Age %{x}<br>Scans: %{y:,}<extra></extra>",
        ),
        secondary_y=True,
    )
    for col in AGE_LINE_COLS:
        legend_name = algo_legend_names[col]
        fig.add_trace(
            go.Scatter(
                x=age_df["age_group"],
                y=age_df[col],
                name=legend_name,
                mode="lines+markers",
                line=dict(color=COLOR_MAP[col]),
                marker=dict(color=COLOR_MAP[col]),
                hovertemplate=f"{legend_name}<br>%{{x}}: %{{y:.1f}}%<extra></extra>",
            ),
            secondary_y=False,
        )
    fig.update_layout(
        title="Prevalence & Algorithm Accuracy by Age Group",
        legend_title_text="",
        xaxis=dict(
            type="category",
            title="Age Group (years)",
            categoryorder="array",
            categoryarray=[str(label) for label in FINE_AGE_LABELS],
            tickangle=-45,
        ),
        barmode="overlay",
        margin=dict(b=80),
    )
    fig.update_yaxes(title_text="Rate (%)", secondary_y=False, range=[0, 100])
    fig.update_yaxes(title_text="Scan count", secondary_y=True, showgrid=False)
    return fig

